# 04 - Decision Tree\n\nArvore de decisao em Spark MLlib, respeitando a restricao do enunciado de nao usar ensembles.

In [ ]:
import time\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nimport seaborn as sns\nfrom pyspark.sql import SparkSession, functions as F\nfrom pyspark.ml import Pipeline\nfrom pyspark.ml.evaluation import RegressionEvaluator\nfrom pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer\nfrom pyspark.ml.regression import DecisionTreeRegressor\n\nSEED = 42\nSPLIT_DATE = '2023-06-01'\nSILVER_PATH = '/data/silver/trips_silver'\nMODEL_OUTPUT = '/models/decision_tree'\nRESULTS_PATH = Path('/results/model_comparison.csv')\nIMPORTANCE_PLOT = Path('/results/feature_importance_decision_tree.png')\nTARGET_COL = 'base_passenger_fare'\n\nNUMERIC_COLS = [\n    'trip_miles', 'trip_time', 'wait_time_sec', 'speed_mph', 'pickup_hour',\n    'pickup_dow', 'pickup_month_num', 'is_weekend', 'is_rush_hour',\n    'is_late_night', 'pickup_airport', 'dropoff_airport', 'same_borough',\n    'shared_req', 'wav_req'\n]\nCATEGORICAL_COLS = ['hvfhs_license_num', 'pu_borough', 'do_borough']\n\nspark = (SparkSession.builder\n    .appName('nyc-rideshare-decision-tree')\n    .master('spark://spark-master:7077')\n    .config('spark.executor.memory', '3g')\n    .config('spark.driver.memory', '4g')\n    .config('spark.sql.shuffle.partitions', '200')\n    .getOrCreate())\n\nspark.sparkContext.setLogLevel('WARN')\nsns.set_theme(style='whitegrid')

In [ ]:
silver_df = spark.read.parquet(SILVER_PATH)
silver_df.createOrReplaceTempView('trips_silver')

# Split temporal via Spark SQL puro (regra do projeto: data prep em SQL, ML pipeline em Python).
train_df = spark.sql(f"""
    SELECT * FROM trips_silver
    WHERE pickup_datetime < TIMESTAMP '{SPLIT_DATE}'
""").cache()
test_df = spark.sql(f"""
    SELECT * FROM trips_silver
    WHERE pickup_datetime >= TIMESTAMP '{SPLIT_DATE}'
""").cache()

train_rows = train_df.count()
test_rows = test_df.count()
if train_rows == 0 or test_rows == 0:
    raise ValueError('O split temporal exige dados antes e depois de 2023-06-01. Um unico mes como 2023-08 nao basta para treinar e avaliar.')

train_rows, test_rows

In [ ]:
indexers = [
    StringIndexer(inputCol=col_name, outputCol=f'{col_name}_idx', handleInvalid='keep')
    for col_name in CATEGORICAL_COLS
]
feature_cols = NUMERIC_COLS + [f'{col_name}_idx' for col_name in CATEGORICAL_COLS]
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
# VectorIndexer marca features com <= maxCategories valores unicos como categoricas.
# Cobre license_num (4), pu/do_borough (6 boroughs + Unknown via handleInvalid='keep' = 7),
# alem de marcar as binarias (is_weekend, is_rush_hour, etc.) como categoricas - efeito benigno.
vector_indexer = VectorIndexer(inputCol='features_raw', outputCol='features', maxCategories=8)
dt = DecisionTreeRegressor(
    labelCol=TARGET_COL,
    featuresCol='features',
    predictionCol='prediction',
    maxDepth=8,
    maxBins=256,
    seed=SEED
)

pipeline = Pipeline(stages=indexers + [assembler, vector_indexer, dt])

In [ ]:
start_time = time.perf_counter()\ndt_model = pipeline.fit(train_df)\ntrain_seconds = time.perf_counter() - start_time\n\ndt_model.write().overwrite().save(MODEL_OUTPUT)\nprint(f'Modelo salvo em {MODEL_OUTPUT} | treino: {train_seconds:.2f}s')

In [ ]:
predictions = dt_model.transform(test_df).cache()\nevaluators = {\n    'rmse': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='rmse'),\n    'mae': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='mae'),\n    'r2': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='r2')\n}\nmetrics = {name: evaluator.evaluate(predictions) for name, evaluator in evaluators.items()}\nmetrics

In [ ]:
predictions.createOrReplaceTempView('predictions_dt')\n\nspark.sql("""\nSELECT\n    pu_borough,\n    ROUND(SQRT(AVG(POWER(base_passenger_fare - prediction, 2))), 4) AS rmse,\n    ROUND(AVG(ABS(base_passenger_fare - prediction)), 4) AS mae,\n    COUNT(*) AS rows\nFROM predictions_dt\nGROUP BY 1\nORDER BY rmse DESC\n""").show(truncate=False)\n\nspark.sql("""\nSELECT\n    CASE\n        WHEN trip_miles < 2 THEN '0-2 mi'\n        WHEN trip_miles < 5 THEN '2-5 mi'\n        WHEN trip_miles < 10 THEN '5-10 mi'\n        ELSE '10+ mi'\n    END AS distance_bucket,\n    ROUND(SQRT(AVG(POWER(base_passenger_fare - prediction, 2))), 4) AS rmse,\n    ROUND(AVG(ABS(base_passenger_fare - prediction)), 4) AS mae,\n    COUNT(*) AS rows\nFROM predictions_dt\nGROUP BY 1\nORDER BY distance_bucket\n""").show(truncate=False)

In [ ]:
tree_stage = dt_model.stages[-1]\nimportance_pdf = pd.DataFrame({\n    'feature': feature_cols,\n    'importance': list(tree_stage.featureImportances)\n}).sort_values('importance', ascending=False)\n\nplt.figure(figsize=(10, 6))\nsns.barplot(data=importance_pdf.head(15), x='importance', y='feature')\nplt.title('Top 15 importancias - Decision Tree')\nplt.tight_layout()\nplt.savefig(IMPORTANCE_PLOT, dpi=150, bbox_inches='tight')\nplt.show()\n\nimportance_pdf.head(15)

In [ ]:
results_df = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame(columns=['model', 'rmse', 'mae', 'r2', 'train_seconds', 'notes'])\nresults_df = results_df[results_df['model'] != 'decision_tree']\nresults_df = pd.concat([\n    results_df,\n    pd.DataFrame([{\n        'model': 'decision_tree',\n        'rmse': metrics['rmse'],\n        'mae': metrics['mae'],\n        'r2': metrics['r2'],\n        'train_seconds': train_seconds,\n        'notes': 'DecisionTreeRegressor maxDepth=8 maxBins=256'\n    }])\n], ignore_index=True)\nresults_df.to_csv(RESULTS_PATH, index=False)\nresults_df

In [ ]:
spark.stop()